# Cigna RAG Demo 2 — 전체 파이프라인

**SKN24 3차 프로젝트 · 4팀 · 담당: 김은우**

이 노트북은 Cigna Global 국제 보험 문서 전체(Customer Guide, Policy Rules, Benefits Summary)를 벡터 DB에 적재하고,
멀티홉·메타데이터 필터·Term Locking 등을 적용해 RAG 파이프라인의 성능을 점검한다.

---

**참고 기법:**
- `01_rag_workflow` — 기본 LCEL 체인 (PromptTemplate + RunnablePassthrough + StrOutputParser)
- `02-8_metadata_filtering` — SelfQueryRetriever (플랜별 필터)
- `02-2_bm25_dense_retrieval` + `02-3_rrf` — BM25+Dense 하이브리드 + RRF
- `02-9_multihop` — 멀티홉 검색 체인

## Section 0 · 환경 설정

In [2]:
# # 가상환경 확인
# import sys
# import torch
# import transformers

# print(f"현재 경로: {sys.executable}")  # SKN_3rd_Project 경로가 나와야 함
# print(f"Torch 버전: {torch.__version__}")
# print(f"Transformers 버전: {transformers.__version__}") # 4.44.2 여부 확인

In [3]:
# !pip install -q langchain langchain-community langchain-openai langchain-chroma
# !pip install -q chromadb openai tiktoken
# !pip install -q pypdf pdfplumber rank_bm25
# !pip install -q sentence-transformers  # BAAI/bge-m3
# print('✅ 설치 완료')

In [4]:
import os, json, re
from pathlib import Path
from typing import List, Dict, Any

# OpenAI API 키 설정
# os.environ['OPENAI_API_KEY'] = 'sk-...'
from dotenv import load_dotenv
load_dotenv()

# ── 경로 설정 ──────────────────────────────────────
BASE_DIR = Path('.')  # 노트북 위치 기준
CIGNA_DIR = BASE_DIR / 'Cigna'

# 폴더 존재 확인
for sub in ['Customer_Guide', 'Policy_Rules', 'Benefits_Summary']:
    p = CIGNA_DIR / sub
    files = list(p.glob('*.pdf')) if p.exists() else []
    print(f'{sub}: {len(files)} PDFs')

Customer_Guide: 4 PDFs
Policy_Rules: 5 PDFs
Benefits_Summary: 3 PDFs


---
## Section 1 · 데이터 로드 — 모든 Cigna PDF + 메타데이터

각 PDF에 `source_type`, `doc_version`, `is_latest`, `plan_type`, `file_name` 메타데이터를 부여한다.

> **pdfplumber 사용 이유:** Benefits Summary는 Silver/Gold/Platinum 보장 금액이 **표(table)** 형태로 저장돼 있어
> 일반 PyPDF 파서로는 텍스트가 깨진다. pdfplumber는 표 구조를 보존해 청크 품질을 높인다.

In [5]:
import pdfplumber
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document                               # Document 객체 등이 필요한 경우
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage    # 메시지 객체가 필요한 경우
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── PDF 메타데이터 정의 ─────────────────────────────
PDF_META = [
    # Customer Guide
    {'path': 'Cigna/Customer_Guide/200008 CGHO Customer Guide EN_05_2019.pdf',
     'source_type': 'customer_guide', 'doc_version': '2019', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Customer_Guide/591048 CGHO Customer Guide EN_05_2022.pdf',
     'source_type': 'customer_guide', 'doc_version': '2022', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Customer_Guide/591048-cgho-customer-guide-en_05_2023.pdf',
     'source_type': 'customer_guide', 'doc_version': '2023', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Customer_Guide/Cigna-Global-Health-Options-Customer-Guide_02_2026.pdf',
     'source_type': 'customer_guide', 'doc_version': '2026', 'is_latest': True, 'plan_type': 'all'},
    # Policy Rules
    {'path': 'Cigna/Policy_Rules/200008 CGHO Customer Guide EN_05_2019.pdf',
     'source_type': 'policy_rules', 'doc_version': '2019', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Policy_Rules/CGHO Policy Rules CGIC NA_EN_05_2023.pdf',
     'source_type': 'policy_rules', 'doc_version': '2023', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Policy_Rules/CGHO Policy Rules CGIC_EN_02_2024.pdf',
     'source_type': 'policy_rules', 'doc_version': '2024', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Policy_Rules/CGHO Policy Rules CGIC_EN_02_2025.pdf',
     'source_type': 'policy_rules', 'doc_version': '2025', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Policy_Rules/CGHP Policy Rules CGIC EN 02_2026.pdf',
     'source_type': 'policy_rules', 'doc_version': '2026', 'is_latest': True, 'plan_type': 'all'},
    # Benefits Summary (pdfplumber로 표 추출)
    {'path': 'Cigna/Benefits_Summary/591116 Cigna_Global_International_Health_Plans_Benefits_Summary_USD_EN_0523.pdf',
     'source_type': 'benefits_summary', 'doc_version': '2023-05', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Benefits_Summary/591116 Cigna Global Benefits Summary USD_EN_0924.pdf',
     'source_type': 'benefits_summary', 'doc_version': '2024-09', 'is_latest': False, 'plan_type': 'all'},
    {'path': 'Cigna/Benefits_Summary/591116-cigna-global-benefits-summary-usd_en_02_2025.pdf',
     'source_type': 'benefits_summary', 'doc_version': '2025-02', 'is_latest': True, 'plan_type': 'all'},
]
print(f'총 {len(PDF_META)}개 PDF 정의됨')

총 12개 PDF 정의됨


In [6]:
def load_pdf_with_pdfplumber(meta: dict) -> List[Document]:
    """Benefits Summary 등 표가 많은 PDF에 pdfplumber를 사용한다."""
    docs = []
    try:
        with pdfplumber.open(meta['path']) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                # 일반 텍스트
                text = page.extract_text() or ''
                # 표 추출 → 탭 구분 텍스트로 변환
                tables = page.extract_tables()
                table_texts = []
                for table in tables:
                    for row in table:
                        row_clean = [str(cell).strip() if cell else '' for cell in row]
                        table_texts.append(' | '.join(row_clean))
                combined = text + ('\n\n[TABLE]\n' + '\n'.join(table_texts) if table_texts else '')
                if combined.strip():
                    docs.append(Document(
                        page_content=combined,
                        metadata={
                            **meta,
                            'page': page_num,
                            'file_name': Path(meta['path']).name,
                        }
                    ))
    except Exception as e:
        print(f'  ⚠ pdfplumber 오류 {meta["path"]}: {e}')
    return docs

def load_pdf_with_pypdf(meta: dict) -> List[Document]:
    """일반 텍스트 위주 PDF (Customer Guide, Policy Rules)."""
    docs = []
    try:
        loader = PyPDFLoader(meta['path'])
        pages = loader.load()
        for page in pages:
            page.metadata.update({
                **meta,
                'file_name': Path(meta['path']).name,
            })
            docs.append(page)
    except Exception as e:
        print(f'  ⚠ PyPDF 오류 {meta["path"]}: {e}')
    return docs

In [7]:
# ── 전체 PDF 로드 ──────────────────────────────────
all_raw_docs: List[Document] = []

for meta in PDF_META:
    path = Path(meta['path'])
    if not path.exists():
        print(f'  ⚠ 파일 없음: {path}')
        continue
    # Benefits Summary는 pdfplumber, 나머지는 PyPDF
    if meta['source_type'] == 'benefits_summary':
        docs = load_pdf_with_pdfplumber(meta)
    else:
        docs = load_pdf_with_pypdf(meta)
    all_raw_docs.extend(docs)
    print(f'  ✅ [{meta["source_type"]} {meta["doc_version"]}] {len(docs)} pages 로드')

print(f'\n총 {len(all_raw_docs)} pages 로드 완료')

  ✅ [customer_guide 2019] 44 pages 로드
  ✅ [customer_guide 2022] 43 pages 로드
  ✅ [customer_guide 2023] 44 pages 로드
  ✅ [customer_guide 2026] 56 pages 로드
  ✅ [policy_rules 2019] 44 pages 로드
  ✅ [policy_rules 2023] 18 pages 로드
  ✅ [policy_rules 2024] 20 pages 로드
  ✅ [policy_rules 2025] 22 pages 로드
  ✅ [policy_rules 2026] 26 pages 로드
  ✅ [benefits_summary 2023-05] 5 pages 로드
  ✅ [benefits_summary 2024-09] 5 pages 로드
  ✅ [benefits_summary 2025-02] 5 pages 로드

총 332 pages 로드 완료


In [8]:
# ── 청킹 (01_rag_workflow 스타일) ──────────────────
# 참고: 01_rag_workflow.ipynb — RecursiveCharacterTextSplitter, chunk_size=500, overlap=100
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

all_chunks: List[Document] = text_splitter.split_documents(all_raw_docs)
print(f'청크 수: {len(all_chunks)}')
print('\n── 샘플 청크 ──')
print(all_chunks[0].page_content[:300])
print('메타데이터:', all_chunks[0].metadata)

청크 수: 2261

── 샘플 청크 ──
Global Health Options
CUSTOMER GUIDE
Everything you need to know about your plan
메타데이터: {'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 2017 (Windows)', 'creationdate': '2019-05-23T06:57:39+01:00', 'moddate': '2019-05-24T10:38:04+01:00', 'trapped': '/False', 'source': 'Cigna/Customer_Guide/200008 CGHO Customer Guide EN_05_2019.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'path': 'Cigna/Customer_Guide/200008 CGHO Customer Guide EN_05_2019.pdf', 'source_type': 'customer_guide', 'doc_version': '2019', 'is_latest': False, 'plan_type': 'all', 'file_name': '200008 CGHO Customer Guide EN_05_2019.pdf'}


---
## Section 2 · 임베딩 모델 비교

| 모델 | 특징 | 비고 |
|------|------|------|
| `text-embedding-3-small` | OpenAI, 영어 최적화, API 호출 | 기본 옵션 |
| `BAAI/bge-m3` | 다국어(100+), 로컬 실행, HuggingFace | 한국어 질문 → 영어 문서 교차에 강함 |

> **선택 근거:** 시스템 사용자가 한국어·영어 혼용 질문을 하고, 문서는 영어이므로
> 다국어 임베딩(bge-m3)이 언어 갭을 줄여 검색 품질을 높일 수 있다.

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# ── 옵션 A: OpenAI text-embedding-3-small ─────────
embed_openai = OpenAIEmbeddings(
    model='text-embedding-3-small',
)
print('OpenAI 임베딩 초기화 완료')

# ── 옵션 B: BAAI/bge-m3 (다국어, 로컬) ────────────
# 첫 실행 시 약 2~3GB 모델 다운로드 필요
embed_bge = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'},  # GPU 있으면 'cuda'
    encode_kwargs={'normalize_embeddings': True},
)
print('BGE-M3 임베딩 초기화 완료')

OpenAI 임베딩 초기화 완료


/var/folders/sk/_9bn_nr9727cxqwf9sxykqxr0000gn/T/ipykernel_88440/1135792337.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed_bge = HuggingFaceEmbeddings(


BGE-M3 임베딩 초기화 완료


In [10]:
# ── 활성 임베딩 선택 ───────────────────────────────
# 'openai' 또는 'bge' 중 선택
EMBEDDING_MODE = 'bge'  # 변경 가능

active_embeddings = embed_openai if EMBEDDING_MODE == 'openai' else embed_bge
print(f'활성 임베딩: {EMBEDDING_MODE}')

활성 임베딩: bge


In [11]:
import numpy as np
import torch

# 1. Numpy가 정상 로드되는지
print(f"Numpy 버전: {np.__version__}") 

# 2. Torch와 Numpy가 대화가 되는지 (에러나면 안 됨)
test_tensor = torch.randn(2, 2)
test_numpy = test_tensor.numpy()
print("✅ Numpy-Torch 연결 성공!")

Numpy 버전: 1.26.4
✅ Numpy-Torch 연결 성공!


In [12]:
# ── Chroma 벡터스토어 구축 ─────────────────────────
# 최신 문서만 우선 인덱싱 (is_latest=True)
latest_chunks = [c for c in all_chunks if c.metadata.get('is_latest', False)]
all_version_chunks = all_chunks  # 버전 비교 시 사용

print(f'최신 버전 청크: {len(latest_chunks)}')
print(f'전체 버전 청크: {len(all_version_chunks)}')

# 최신 문서 벡터스토어
vectorstore_latest = Chroma.from_documents(
    documents=latest_chunks,
    embedding=active_embeddings,
    collection_name='cigna_latest',
    persist_directory='./chroma_cigna_latest',
)

# 전체 버전 벡터스토어 (버전 비교·상충 감지용)
vectorstore_all = Chroma.from_documents(
    documents=all_version_chunks,
    embedding=active_embeddings,
    collection_name='cigna_all',
    persist_directory='./chroma_cigna_all',
)
print('✅ 벡터스토어 구축 완료')

최신 버전 청크: 637
전체 버전 청크: 2261


InvalidArgumentError: Collection expecting embedding with dimension of 1536, got 1024

---
## Section 3 · 리트리버 구성

### 3-A. 기본 Dense 리트리버




In [ ]:
# ── 3-A. 기본 Dense 리트리버 ──────────────────────
retriever_dense = vectorstore_latest.as_retriever(
    search_type='mmr',  # Maximum Marginal Relevance로 다양성 확보
    search_kwargs={'k': 5, 'fetch_k': 20},
)
print('Dense 리트리버 준비 완료')

Dense 리트리버 준비 완료


##### 패키지 문제

결국 langchain-classic 사용
다운그레이드 시 의존성 충돌이 발생했네요. 먼저 원래대로 되돌리고, langchain-classic 패키지를 사용하는 방법을 써보세요.


In [ ]:
# !pip install lark

In [ ]:
# !pip install langchain langchain-community langchain-openai
# !pip install -U langchain langchain-community --quiet
# !pip install -U langchain langchain-openai
# !pip install langchain-community --quiet
# !pip install langchain==1.2.14 langchain-core>=1.2.19 --quiet
!pip show langchain


Name: langchain
Version: 1.2.14
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /opt/miniconda3/envs/SKN_3rd_Project/lib/python3.10/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [ ]:
import langchain
print(langchain.__version__)

import langchain_community.retrievers as r
print(r.__all__)

1.2.14
['AmazonKendraRetriever', 'AmazonKnowledgeBasesRetriever', 'ArceeRetriever', 'ArxivRetriever', 'AskNewsRetriever', 'AzureAISearchRetriever', 'AzureCognitiveSearchRetriever', 'BM25Retriever', 'BreebsRetriever', 'ChaindeskRetriever', 'ChatGPTPluginRetriever', 'CohereRagRetriever', 'DocArrayRetriever', 'DriaRetriever', 'ElasticSearchBM25Retriever', 'EmbedchainRetriever', 'GoogleCloudEnterpriseSearchRetriever', 'GoogleDocumentAIWarehouseRetriever', 'GoogleVertexAIMultiTurnSearchRetriever', 'GoogleVertexAISearchRetriever', 'KayAiRetriever', 'KNNRetriever', 'LlamaIndexGraphRetriever', 'LlamaIndexRetriever', 'MetalRetriever', 'MilvusRetriever', 'NanoPQRetriever', 'NeedleRetriever', 'NeuralDBRetriever', 'OutlineRetriever', 'PineconeHybridSearchRetriever', 'PubMedRetriever', 'QdrantSparseVectorRetriever', 'RememberizerRetriever', 'RemoteLangChainRetriever', 'SVMRetriever', 'TavilySearchAPIRetriever', 'TFIDFRetriever', 'VespaRetriever', 'WeaviateHybridSearchRetriever', 'WebResearchRetriev

### 3-B. 메타데이터 필터 (플랜별)
> 참고: `02-8_metadata_filtering.ipynb` — SelfQueryRetriever + AttributeInfo

In [ ]:
# ── 3-B. 메타데이터 필터 리트리버 ────────────────
# 참고: 02-8_metadata_filtering.ipynb — SelfQueryRetriever
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_openai import ChatOpenAI

metadata_field_info = [
    AttributeInfo(
        name='source_type',
        description='문서 유형. customer_guide / policy_rules / benefits_summary',
        type='string',
    ),
    AttributeInfo(
        name='doc_version',
        description='문서 연도 버전. 예: 2022, 2024, 2025-02',
        type='string',
    ),
    AttributeInfo(
        name='is_latest',
        description='최신 버전 여부 (True/False)',
        type='boolean',
    ),
    AttributeInfo(
        name='plan_type',
        description='보험 플랜 종류. Silver / Gold / Platinum / all',
        type='string',
    ),
]

document_content_description = \
    'Cigna Global 국제 건강보험 약관·고객 가이드·혜택 요약 문서'

llm_filter = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

retriever_selfquery = SelfQueryRetriever.from_llm(
    llm=llm_filter,
    vectorstore=vectorstore_all,  # 전체 버전 포함
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True,
)
print('SelfQueryRetriever 준비 완료')

SelfQueryRetriever 준비 완료


### 3-C. BM25 + Dense 하이브리드 (RRF)
> 참고: `02-2_bm25_dense_retrieval.ipynb`, `02-3_rrf.ipynb`

In [ ]:
# ── 3-C. BM25 + Dense 하이브리드 (RRF) ───────────
# 참고: 02-2_bm25_dense_retrieval.ipynb + 02-3_rrf.ipynb
from rank_bm25 import BM25Okapi

# BM25 인덱스 구성
bm25_corpus = [c.page_content.lower().split() for c in latest_chunks]
bm25 = BM25Okapi(bm25_corpus)

def bm25_search(query: str, k: int = 5) -> List[Document]:
    """BM25 키워드 기반 검색 (02-2 스타일)."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [latest_chunks[i] for i in top_indices]

def rrf_rank(bm25_list: List[Document], dense_list: List[Document], k: int = 60) -> List[Document]:
    """Reciprocal Rank Fusion (02-3 스타일)."""
    scores: Dict[str, float] = {}
    doc_map: Dict[str, Document] = {}
    for rank, doc in enumerate(bm25_list, start=1):
        key = doc.page_content[:80]
        scores[key] = scores.get(key, 0) + 1 / (k + rank)
        doc_map[key] = doc
    for rank, doc in enumerate(dense_list, start=1):
        key = doc.page_content[:80]
        scores[key] = scores.get(key, 0) + 1 / (k + rank)
        doc_map[key] = doc
    sorted_keys = sorted(scores, key=scores.get, reverse=True)
    return [doc_map[k] for k in sorted_keys]

def hybrid_retriever(query: str, k: int = 5) -> List[Document]:
    """BM25 + Dense 하이브리드 검색."""
    bm25_results = bm25_search(query, k=k*2)
    dense_results = retriever_dense.invoke(query)
    merged = rrf_rank(bm25_results, dense_results)
    return merged[:k]

print('Hybrid Retriever (BM25 + Dense + RRF) 준비 완료')

Hybrid Retriever (BM25 + Dense + RRF) 준비 완료


---
## Section 4 · RAG 체인 — Term Locking + 출처 인용

**Term Locking**이란 시스템 프롬프트에 보험 전문 용어의 번역 고정 테이블을 삽입해,
답변 전반에서 일관된 용어를 사용하게 하는 기법이다.
예: `Deductible → 공제액(Deductible)`, `Co-insurance → 공동부담률(Co-insurance)`

> 참고: `01_rag_workflow.ipynb` — `format_docs`, `RunnablePassthrough`, `StrOutputParser`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Term Locking 용어 테이블 ────────────────────────
TERM_LOCK_TABLE = """
=== 보험 용어 번역 고정 테이블 (Term Locking) ===
아래 용어는 반드시 아래 형식으로만 표기할 것:
- Deductible → 공제액(Deductible)
- Co-insurance / Cost Share → 공동부담률(Co-insurance)
- Copay → 정액 본인부담(Copay)
- Out-of-Pocket Maximum → 최대 본인부담금(OOP Max)
- Prior Approval / Pre-authorization → 사전 승인(Prior Approval)
- In-network → 네트워크 내(In-network)
- Out-of-network → 네트워크 외(Out-of-network)
- Inpatient → 입원(Inpatient)
- Outpatient → 외래(Outpatient)
- Mental Health Care → 정신건강 케어(Mental Health Care)
- Psychotherapy → 심리치료(Psychotherapy)
- Free Look Period → 청약 철회 기간(Free Look Period)
"""

# ── format_docs (01_rag_workflow 스타일) ───────────
def format_docs(docs: List[Document]) -> str:
    """검색된 청크를 출처 정보와 함께 포매팅."""
    formatted = []
    for i, doc in enumerate(docs, start=1):
        meta = doc.metadata
        source_label = f"[{i}] {meta.get('source_type','?')} {meta.get('doc_version','?')} p.{meta.get('page','?')}"
        formatted.append(f"{source_label}\n{doc.page_content}")
    return '\n\n'.join(formatted)

print('format_docs 준비 완료')

format_docs 준비 완료


In [ ]:
# ── RAG 프롬프트 템플릿 ─────────────────────────────
RAG_TEMPLATE = """
당신은 Cigna Global 국제 건강보험 전문 안내 어시스턴트입니다.
아래 [참고 문서]를 근거로 질문에 답하세요.

{term_lock}

규칙:
1. 반드시 [참고 문서]에 있는 내용만 답변하세요.
2. 답변 말미에 근거 문서를 [출처: 문서유형 버전 p.페이지] 형식으로 인용하세요.
3. 문서에 없는 정보는 '문서에서 확인할 수 없습니다'라고 답하세요.
4. 보험 추천·가입 권유는 하지 않습니다.
5. 답변은 한국어로 작성하되, 위 Term Locking 표에 따른 전문 용어를 사용하세요.

[참고 문서]
{context}

[질문]
{question}
"""

rag_prompt = PromptTemplate(
    input_variables=['context', 'question', 'term_lock'],
    template=RAG_TEMPLATE,
)

# ── LLM (gpt-4.1-mini) ────────────────────────────
llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0)

# ── LCEL 체인 (01_rag_workflow 스타일) ──────────────
# 참고: 01_rag_workflow.ipynb — RunnablePassthrough + PromptTemplate + LLM + StrOutputParser
def make_rag_chain(retriever):
    chain = (
        {
            'context': retriever | format_docs,
            'question': RunnablePassthrough(),
            'term_lock': lambda _: TERM_LOCK_TABLE,
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    return chain

# 기본 체인 (Dense 리트리버)
rag_chain_dense   = make_rag_chain(retriever_dense)
# Hybrid 체인 (BM25 + Dense + RRF)
class HybridRetrieverWrapper:
    def invoke(self, query): return hybrid_retriever(query)
# rag_chain_hybrid  = make_rag_chain(HybridRetrieverWrapper())

# 제미나이 해결법
from langchain_core.runnables import RunnableLambda
rag_chain_hybrid = make_rag_chain(RunnableLambda(HybridRetrieverWrapper().invoke))

print('RAG 체인 (Dense + Hybrid) 준비 완료')

RAG 체인 (Dense + Hybrid) 준비 완료


In [ ]:
# 관련 핵심 패키지들을 모두 최신으로 업데이트하여 내부 참조를 일치시킵니다 ㅋ
# !pip install -U langchain-core langchain-openai langchain-community langchain

  Using cached langchain_core-1.2.23-py3-none-any.whl.metadata (4.4 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain-1.2.14-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_core-1.2.23-py3-none-any.whl (506 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached langchain_text_splitters-1.1.1-py3-none-any.whl (35 kB)
Using cached langchain-1.2.14-py3-none-any.whl (112 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.83
    Uninstalling langchain-core-0.3.83:
      Successfully uninstalled langchain-core-0.3.83
  Attempting uninstall: langchain-text-splitters 0/4 [langchain-core]
    Found existing installation: langchain-text-splitters 0.3.11angchain-core]
    Uninstalling langchain-text-splitters-0.3.11:0/4 [langchain-core]
      Successfully uninstalled langchain-text-s

In [ ]:
# ── 단일 질문 실행 헬퍼 ────────────────────────────
def ask(question: str, mode: str = 'hybrid', verbose: bool = True) -> str:
    """RAG 질문 실행. mode: 'dense' | 'hybrid' | 'selfquery'"""
    if mode == 'dense':
        answer = rag_chain_dense.invoke(question)
    elif mode == 'selfquery':
        docs = retriever_selfquery.invoke(question)
        context = format_docs(docs)
        answer = (rag_prompt | llm | StrOutputParser()).invoke(
            {'context': context, 'question': question, 'term_lock': TERM_LOCK_TABLE}
        )
    else:  # hybrid (default)
        answer = rag_chain_hybrid.invoke(question)
    if verbose:
        print(f'[Q] {question}')
        print(f'[A] {answer}')
        print('-' * 60)
    return answer

# 빠른 동작 확인
_ = ask('Cigna Global Silver 플랜의 공제액(Deductible) 옵션은?', mode='hybrid')

[Q] Cigna Global Silver 플랜의 공제액(Deductible) 옵션은?
[A] Cigna Global Silver 플랜의 공제액(Deductible) 옵션에 관한 구체적인 내용은 문서에서 확인할 수 없습니다.

[출처: customer_guide 2026 p.48]
------------------------------------------------------------


---
## Section 5 · 버전 간 상충 감지 (Conflict Detection)

동일 질문에 대해 2022년 버전과 최신 버전의 답변을 비교해,
약관이 변경된 부분을 자동으로 표시한다.

> **발표 포인트:** PDF 교체만으로 약관 업데이트가 즉시 반영되는 시연에 활용.

In [ ]:
# ── 버전별 리트리버 ────────────────────────────────
def make_version_retriever(version: str, source_type: str = None):
    """특정 버전 문서만 검색하는 리트리버를 반환."""
    filter_dict = {'doc_version': version}
    if source_type:
        filter_dict['source_type'] = source_type
    return vectorstore_all.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 4, 'filter': filter_dict},
    )

def detect_conflict(question: str, old_version: str = '2022', new_version: str = '2026'):
    """두 버전 답변을 비교해 상충 여부를 감지한다."""
    retriever_old = make_version_retriever(old_version)
    retriever_new = make_version_retriever(new_version)

    chain_old = make_rag_chain(retriever_old)
    chain_new = make_rag_chain(retriever_new)

    ans_old = chain_old.invoke(question)
    ans_new = chain_new.invoke(question)

    # GPT에게 두 답변 비교 요청
    compare_prompt = f"""
다음 두 버전의 보험 약관 답변을 비교하고, 변경된 내용이 있으면 구체적으로 설명하세요.
변경이 없으면 '변경 없음'이라고 답하세요.

=== {old_version}년 버전 답변 ===
{ans_old}

=== {new_version}년 버전 답변 ===
{ans_new}

비교 결과:
"""
    diff = llm.invoke(compare_prompt).content

    print(f'[질문] {question}')
    print(f'\n[{old_version} 버전]\n{ans_old}')
    print(f'\n[{new_version} 버전]\n{ans_new}')
    print(f'\n[변경 감지]\n{diff}')
    return {'old': ans_old, 'new': ans_new, 'diff': diff}

print('버전 비교 함수 준비 완료')

버전 비교 함수 준비 완료


In [ ]:
# ── 버전 비교 시연: 청약 철회 기간 ───────────────
# Gemini Q12 연계: 2022 → 2026 버전 변경 사항 감지
conflict_result = detect_conflict(
    question='Free Look Period는 며칠인가요?',
    old_version='2022',
    new_version='2026',
)

[질문] Free Look Period는 며칠인가요?

[2022 버전]
문서에서 Free Look Period(청약 철회 기간)에 대한 정보는 확인할 수 없습니다.

[출처: customer_guide 2022]

[2026 버전]
Free Look Period는 14일입니다.

[출처: policy_rules 2026 p.6]

[변경 감지]
비교 결과:  
2022년 버전에서는 Free Look Period(청약 철회 기간)에 대한 정보가 없다고 명시되어 있으나, 2026년 버전에서는 청약 철회 기간이 14일로 구체적으로 명시되어 있습니다. 따라서 2026년 버전에서 청약 철회 기간에 관한 내용이 새롭게 추가된 변경 사항이 있습니다.


---
## Section 6 · Multi-hop 검색 체인

복합 질문(예: deductible 계산 + 플랜 적용 + 네트워크 조건)은 단일 검색으로 해결이 어렵다.
멀티홉 체인은 첫 검색 결과를 바탕으로 후속 검색을 자동 생성한다.

> 참고: `02-9_multihop.ipynb` — `multihop_search(question, max_hop=3)`

In [ ]:
# 참고: 02-9_multihop.ipynb
def multihop_search(question: str, max_hop: int = 3, verbose: bool = False) -> List[Document]:
    """멀티홉 검색: 이전 컨텍스트를 활용해 후속 쿼리 자동 생성."""
    accumulated_docs: List[Document] = []
    current_query = question

    for hop in range(max_hop):
        new_docs = hybrid_retriever(current_query, k=3)
        accumulated_docs.extend(new_docs)
        if verbose:
            print(f'  [Hop {hop+1}] 쿼리: {current_query[:60]}')
            print(f'           검색 결과: {len(new_docs)}개')

        if hop < max_hop - 1:
            # 현재까지 수집된 컨텍스트로 다음 검색어 생성
            context_so_far = format_docs(accumulated_docs[:6])
            follow_up_prompt = f"""
원래 질문: {question}
현재 수집된 정보:
{context_so_far[:800]}

아직 답변에 필요한 정보가 있다면 추가로 검색해야 할 핵심 키워드/문장을 한 줄로 작성하세요.
충분하면 'DONE'이라고만 답하세요.
"""
            next_query = llm.invoke(follow_up_prompt).content.strip()
            if next_query.upper() == 'DONE' or len(next_query) < 3:
                if verbose: print(f'  [Hop {hop+1}] 검색 완료 (DONE)')
                break
            current_query = next_query

    # 중복 제거
    seen = set()
    unique_docs = []
    for doc in accumulated_docs:
        key = doc.page_content[:80]
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)
    return unique_docs

def ask_multihop(question: str) -> str:
    """멀티홉 검색 + RAG 답변."""
    docs = multihop_search(question, max_hop=3, verbose=True)
    context = format_docs(docs)
    answer = (rag_prompt | llm | StrOutputParser()).invoke(
        {'context': context, 'question': question, 'term_lock': TERM_LOCK_TABLE}
    )
    print(f'\n[멀티홉 답변]\n{answer}')
    return answer

print('Multi-hop 체인 준비 완료')

Multi-hop 체인 준비 완료


In [ ]:
# ── 멀티홉 시연: Gemini Q7 (deductible + cost share 계산) ──
_ = ask_multihop(
    'Silver 플랜에서 공제액(Deductible) $500, 공동부담률(Co-insurance) 20%를 선택했을 때, '
    '병원비가 $1,000이라면 보험에서 실제로 지급하는 금액은 얼마인가?'
)

  [Hop 1] 쿼리: Silver 플랜에서 공제액(Deductible) $500, 공동부담률(Co-insurance) 20%를 선
           검색 결과: 3개
  [Hop 2] 쿼리: Silver 플랜 공제액 $500, 공동부담률 20%일 때 $1,000 병원비에 대한 보험 지급액 계산 방법
           검색 결과: 3개
  [Hop 3] 쿼리: Silver 플랜 공제액 및 공동부담률 계산 예시
           검색 결과: 3개

[멀티홉 답변]
Silver 플랜에서 공제액(Deductible)이 $500이고 공동부담률(Co-insurance)이 20%일 때, 병원비가 $1,000인 경우 보험금 지급액을 계산하면 다음과 같습니다.

1. 먼저 공제액 $500를 본인이 부담합니다.
2. 남은 금액 $1,000 - $500 = $500에 대해 공동부담률 20%를 적용하면, 본인이 부담하는 금액은 $500 × 20% = $100입니다.
3. 따라서 본인이 부담하는 총액은 공제액 $500 + 공동부담금 $100 = $600입니다.
4. 보험금 지급액은 총 병원비 $1,000에서 본인 부담금 $600을 뺀 $400입니다.

즉, 보험에서 실제로 지급하는 금액은 $400입니다.

[출처: customer_guide 2026 p.49]


In [15]:
# hybird와 비교
# ── 멀티홉 시연: Gemini Q7 (deductible + cost share 계산) ──
_ = ask('Silver 플랜에서 공제액(Deductible) $500, 공동부담률(Co-insurance) 20%를 선택했을 때, '
    '병원비가 $1,000이라면 보험에서 실제로 지급하는 금액은 얼마인가?')


NameError: name 'ask' is not defined

---
## Section 7 · 테스트 질문셋 전체 실행

총 31개 질문 (Gemini 12 + Perplexity 12 + 기획서 시나리오 7)을 실행하고
결과를 DataFrame으로 저장한다.

In [ ]:
# ── 테스트 질문셋 정의 ──────────────────────────────
TEST_QUESTIONS = [
    # ── Gemini 12문항 ──
    {'id': 'G01', 'level': 'Low',    'source': 'Gemini',
     'question': 'Cigna Healthcare의 공제액(Deductible) 옵션 중 가장 높은 금액은 얼마인가요?'},
    {'id': 'G02', 'level': 'Low',    'source': 'Gemini',
     'question': 'International Vision & Dental 옵션에서 Silver 플랜 기준 정기 치과 진료(Routine dental care)의 연간 한도는 얼마인가요?'},
    {'id': 'G03', 'level': 'Low',    'source': 'Gemini',
     'question': '보험 증권(Certificate of Insurance)에 기재된 Start date의 정의는 무엇인가요?'},
    {'id': 'G04', 'level': 'Medium', 'source': 'Gemini',
     'question': '입원 치료 전에 반드시 사전 승인(Prior Approval)을 받아야 하나요? 받지 않으면 어떤 불이익이 있나요?'},
    {'id': 'G05', 'level': 'Medium', 'source': 'Gemini',
     'question': '보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인가요?'},
    {'id': 'G06', 'level': 'Medium', 'source': 'Gemini',
     'question': '거주 국가가 변경되었을 때 Cigna에 언제까지 알려야 하나요?'},
    {'id': 'G07', 'level': 'High',   'source': 'Gemini',
     'question': 'Silver 플랜에서 공제액(Deductible) $500, 공동부담률(Co-insurance) 20% 선택 시 병원비 $1,000에 대해 실제 환급받을 금액은 얼마인가요?'},
    {'id': 'G08', 'level': 'High',   'source': 'Gemini',
     'question': '태국에서 오토바이 사고가 발생했을 때 위험 활동(Hazardous Activities) 관련 보장 제외 조항이 적용될 수 있나요?'},
    {'id': 'G09', 'level': 'High',   'source': 'Gemini',
     'question': '현재 임신 중이고 6개월 뒤 태어날 아이를 추가하고 싶습니다. 의료 심사(Full Medical Underwriting) 없이 추가 가능한 신청 기한은 언제까지인가요?'},
    {'id': 'G10', 'level': 'High',   'source': 'Gemini',
     'question': '심리상담(Mental Health Support) 서비스는 입원 중에만 가능한가요, 아니면 외래(Outpatient)로도 이용 가능한가요?'},
    {'id': 'G11', 'level': 'High',   'source': 'Gemini',
     'question': '한국 거주자로서 보험금 청구 시 앱으로 청구할 수 있나요? 싱가포르 지점 연락처도 알려주세요.'},
    {'id': 'G12', 'level': 'High',   'source': 'Gemini',
     'question': '2022년 Customer Guide와 비교했을 때 최신 버전에서 달라진 디지털 서비스나 주요 변경 사항은 무엇인가요?'},
    # ── Perplexity 12문항 ──
    {'id': 'P01', 'level': 'Low',    'source': 'Perplexity',
     'question': 'Cigna Global Silver 플랜의 연간 공제액(Deductible)은 얼마인가요?'},
    {'id': 'P02', 'level': 'Low',    'source': 'Perplexity',
     'question': 'Gold 플랜에서 외래 정신건강 방문(Outpatient mental health visit) 당 정액 본인부담(Copay)은 얼마인가요?'},
    {'id': 'P03', 'level': 'Low',    'source': 'Perplexity',
     'question': 'Platinum 플랜의 최대 의료비 한도(Annual Maximum Benefit)는 얼마인가요?'},
    {'id': 'P04', 'level': 'Medium', 'source': 'Perplexity',
     'question': '심리상담 예약 시 사전 승인(Prior Approval)이 필요한가요?'},
    {'id': 'P05', 'level': 'Medium', 'source': 'Perplexity',
     'question': '한국에서 Cigna 네트워크 내(In-network) 병원에서 심리상담을 받을 수 있나요?'},
    {'id': 'P06', 'level': 'Medium', 'source': 'Perplexity',
     'question': '원격의료(Telehealth) 상담도 Cigna에서 보장되나요?'},
    {'id': 'P07', 'level': 'Medium', 'source': 'Perplexity',
     'question': '심리상담 50분 세션 청구 시 어떤 서류가 필요한가요?'},
    {'id': 'P08', 'level': 'Medium', 'source': 'Perplexity',
     'question': '보험금 청구(Claim) 제출 기한은 며칠 이내인가요?'},
    {'id': 'P09', 'level': 'High',   'source': 'Perplexity',
     'question': '연 10회 이상 심리치료(Psychotherapy)를 받을 경우 추가 승인이 필요한가요?'},
    {'id': 'P10', 'level': 'High',   'source': 'Perplexity',
     'question': 'Silver 플랜에서 한국 네트워크 외(Out-of-network) 상담소 이용 시 보장 비율은 얼마인가요?'},
    {'id': 'P11', 'level': 'High',   'source': 'Perplexity',
     'question': '정신과 약 처방도 외래 정신건강 케어(Outpatient Mental Health Care)에 포함되어 보장되나요?'},
    {'id': 'P12', 'level': 'High',   'source': 'Perplexity',
     'question': '가족 중 한 명만 의료비를 사용해도 다른 가족의 공제액(Deductible)이 공유되나요?'},
    # ── 기획서 시나리오 7문항 (Cigna 버전) ──
    {'id': 'S01', 'level': 'High',   'source': 'Plan',
     'question': 'Cigna Global International Outpatient 플랜에서 정신건강 케어(Mental Health Care)는 연간 몇 회까지 보장되나요?'},
    {'id': 'S02', 'level': 'Medium', 'source': 'Plan',
     'question': '보험금 청구 시 공제액(Deductible)과 공동부담률(Co-insurance)의 차이는 무엇인가요?'},
    {'id': 'S03', 'level': 'Medium', 'source': 'Plan',
     'question': '심리상담을 받기 전에 Cigna의 사전 승인(Prior Approval)이 반드시 필요한가요?'},
    {'id': 'S04', 'level': 'High',   'source': 'Plan',
     'question': '상담 후 처방된 약(Prescribed drugs) 비용도 Cigna 플랜에서 청구할 수 있나요?'},
    {'id': 'S05', 'level': 'Medium', 'source': 'Plan',
     'question': 'How much is the benefit limit for Mental Health Care in the Cigna Outpatient plan?'},
    {'id': 'S06', 'level': 'High',   'source': 'Plan',
     'question': '법적 문제 때문에 받는 심리 검사(Psychometric testing)도 Cigna 플랜에서 보장되나요?'},
    {'id': 'S07', 'level': 'High',   'source': 'Plan',
     'question': 'Cigna는 국제 보험이므로 모든 심리상담이 무제한으로 보장되나요?'},
]

print(f'총 테스트 질문: {len(TEST_QUESTIONS)}개')
print(f'  Gemini: {sum(1 for q in TEST_QUESTIONS if q["source"]=="Gemini")}개')
print(f'  Perplexity: {sum(1 for q in TEST_QUESTIONS if q["source"]=="Perplexity")}개')
print(f'  기획서 시나리오: {sum(1 for q in TEST_QUESTIONS if q["source"]=="Plan")}개')

총 테스트 질문: 31개
  Gemini: 12개
  Perplexity: 12개
  기획서 시나리오: 7개


In [ ]:
# !pip install ipywidgets tqdm

In [ ]:
import pandas as pd
from tqdm.notebook import tqdm
import time
# from tqdm import tqdm

# ── 전체 테스트 실행 ────────────────────────────────
results = []

for item in tqdm(TEST_QUESTIONS, desc='테스트 진행'):
    qid     = item['id']
    question = item['question']
    level   = item['level']
    source  = item['source']

    # 난이도에 따라 retriever 선택
    # High 난이도 → 멀티홉, Medium → 하이브리드, Low → Dense
    try:
        if level == 'High':
            docs = multihop_search(question, max_hop=3)
            context = format_docs(docs)
            answer = (rag_prompt | llm | StrOutputParser()).invoke(
                {'context': context, 'question': question, 'term_lock': TERM_LOCK_TABLE}
            )
        elif level == 'Medium':
            answer = rag_chain_hybrid.invoke(question)
        else:  # Low
            answer = rag_chain_dense.invoke(question)
        status = 'OK'
    except Exception as e:
        answer = f'ERROR: {e}'
        status = 'ERROR'

    results.append({
        'ID': qid, 'Source': source, 'Level': level,
        'Question': question[:60] + '...' if len(question) > 60 else question,
        'Answer': answer,
        'Status': status,
    })
    time.sleep(0.3)  # API rate limit 방지

df_results = pd.DataFrame(results)
print(f'\n테스트 완료: {len(df_results)}개 / {df_results["Status"].value_counts().to_dict()}')

테스트 진행:   0%|          | 0/31 [00:00<?, ?it/s]


테스트 완료: 31개 / {'OK': 31}


In [ ]:
# ── 결과 확인 ──────────────────────────────────────
pd.set_option('display.max_colwidth', 120)
display(df_results[['ID','Source','Level','Question','Status']])

,ID,Source,Level,Question,Status
0,G01,Gemini,Low,Cigna Healthcare의 공제액(Deductible) 옵션 중 가장 높은 금액은 얼마인가요?,OK
1,G02,Gemini,Low,International Vision & Dental 옵션에서 Silver 플랜 기준 정기 치과 진료(Rou...,OK
2,G03,Gemini,Low,보험 증권(Certificate of Insurance)에 기재된 Start date의 정의는 무엇인가요?,OK
3,G04,Gemini,Medium,입원 치료 전에 반드시 사전 승인(Prior Approval)을 받아야 하나요? 받지 않으면 어떤 불이익이 ...,OK
4,G05,Gemini,Medium,보험 가입 후 취소하고 싶을 때 전액 환불이 가능한 청약 철회 기간(Free Look Period)은 며칠인...,OK
5,G06,Gemini,Medium,거주 국가가 변경되었을 때 Cigna에 언제까지 알려야 하나요?,OK
6,G07,Gemini,High,"Silver 플랜에서 공제액(Deductible) $500, 공동부담률(Co-insurance) 20% 선택...",OK
7,G08,Gemini,High,태국에서 오토바이 사고가 발생했을 때 위험 활동(Hazardous Activities) 관련 보장 제외 조항...,OK
8,G09,Gemini,High,현재 임신 중이고 6개월 뒤 태어날 아이를 추가하고 싶습니다. 의료 심사(Full Medical Underw...,OK
9,G10,Gemini,High,"심리상담(Mental Health Support) 서비스는 입원 중에만 가능한가요, 아니면 외래(Outpat...",OK


In [ ]:
# ── 전체 답변 출력 ─────────────────────────────────
for _, row in df_results.iterrows():
    print(f"[{row['ID']}] ({row['Source']} / {row['Level']})")
    print(f"Q: {row['Question']}")
    print(f"A: {row['Answer']}")
    print('─' * 70)

[G01] (Gemini / Low)
Q: Cigna Healthcare의 공제액(Deductible) 옵션 중 가장 높은 금액은 얼마인가요?
A: Cigna Healthcare의 공제액(Deductible) 옵션 중 가장 높은 금액은 $10,000입니다.  
[출처: customer_guide 2026 p.26]
──────────────────────────────────────────────────────────────────────
[G02] (Gemini / Low)
Q: International Vision & Dental 옵션에서 Silver 플랜 기준 정기 치과 진료(Rou...
A: International Vision & Dental 옵션의 Silver 플랜 기준 정기 치과 진료(Routine dental care)의 연간 한도는 문서에 명확히 금액으로 기재되어 있지 않습니다. 다만, Silver 플랜은 정기 치과 진료에 대해 80% 환급을 제공하며, 3개월 이상 보험 가입 후 치료가 필요한 경우에 적용됩니다. 구체적인 연간 한도 금액은 문서에서 확인할 수 없습니다.

[출처: customer_guide 2026 p.45-46]
──────────────────────────────────────────────────────────────────────
[G03] (Gemini / Low)
Q: 보험 증권(Certificate of Insurance)에 기재된 Start date의 정의는 무엇인가요?
A: 보험 증권(Certificate of Insurance)에 기재된 Start date의 정의는 "이 보험의 보장이 시작되는 날짜"입니다.

[출처: policy_rules 2026 p.24]
──────────────────────────────────────────────────────────────────────
[G04] (Gemini / Medium)
Q: 입원 치료 전에 반드시 사전 승인(Prior Approval)을 받아야 하나요

In [ ]:
# ── 결과 CSV 저장 ──────────────────────────────────
df_results.to_csv('cigna_rag_test_embedbaai_results.csv', index=False, encoding='utf-8-sig')
print('✅ cigna_rag_test_embedbaai_results.csv 저장 완료')

✅ cigna_rag_test_results.csv 저장 완료


---
## Section 8 · 성능 분석 & 다음 단계

### 결과 해석 포인트

**잘 됐을 것으로 예상:**
- Low 난이도 수치 질문 (P01~P03, G01~G03) — Benefits Summary 표가 pdfplumber로 잘 파싱되면 정확도 ↑
- 절차 질문 (G04~G06, P04~P08) — Customer Guide 청크가 풍부하게 들어 있으므로 검색 히트율 높음

**어려울 것으로 예상:**
- 계산 질문 (G07) — LLM이 산술 추론을 직접 해야 함. 문서에서 공식을 찾아도 계산 오류 가능
- 버전 비교 (G12) — 두 버전 청크가 혼재되어 있어야 비교가 가능
- 한국 네트워크 외 보장 비율 (P10) — CGIC 특정 조항 찾기 어려울 수 있음

### 개선 방향 (팀 회의 아젠다)

1. **파싱 품질 개선:** Benefits Summary 표 → pdfplumber 파싱이 제대로 됐는지 raw 텍스트 점검
2. **청크 크기 조정:** `chunk_size=500`이 표 행을 자르면 `chunk_size=800` or 표 단위 청킹 시도
3. **임베딩 비교:** `EMBEDDING_MODE = 'bge'`로 변경 후 동일 질문셋 재실행 → 정확도 차이 측정
4. **언어 교차 테스트:** S05처럼 영어 질문도 포함해 다국어 검색 성능 확인
5. **Reranker 추가:** `cross-encoder/ms-marco-MiniLM-L-6-v2` 등으로 검색 결과 재순위화

### 파이프라인 구조 요약

```
PDF 로드 (pdfplumber + PyPDF)
    ↓
메타데이터 부여 (source_type, doc_version, is_latest, plan_type)
    ↓
청킹 (RecursiveCharacterTextSplitter, chunk_size=500)
    ↓
임베딩 (text-embedding-3-small / BAAI/bge-m3 선택)
    ↓
Chroma 벡터스토어 (latest / all)
    ↓
리트리버 선택:
  - Low  → Dense (MMR)
  - Med  → Hybrid (BM25 + Dense + RRF)
  - High → Multi-hop (3-hop)
    ↓
RAG 체인 (Term Locking + LCEL + gpt-4.1-mini)
    ↓
출처 인용 [source_type version p.page]
```
